# 91장 · 최소 동작 경로 · 방향과 예시

Q01 한 건을 끝까지 보내 보겠습니다. 사용자가 질문을 쓰고 장비 규정 한 줄이 화면에 뜨면 최소 경로를 연결한 것입니다. 같은 방식으로 교체 일정까지 대답해도 되는지는 다른 질문에서 드러납니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.

## 질문 하나를 규정 한 줄에 연결하기

91장의 첫 구현은 장비명과 허용 그룹으로 규정을 찾습니다. 먼저 질문을 넣었을 때 화면에 어떤 답이 돌아오는지 보고, 서버와 모델 호출은 뒤로 미룹니다. 아래 셀은 공개 그룹의 장비 질문 경로를 재현합니다. 응답의 sources에는 아직 문서 ID만 담습니다.



**볼 곳**　‘장비 A 질문’에서 출발해 ‘규정 A’를 거친 뒤 ‘전원 표시등’가 되는 순서를 따라가세요.

**애니메이션의 의미**　장비 A 질문이 규정 A의 전원 표시등 안내로 이어집니다.


### 셀 01 · Q01로 첫 답 받기

A가 들어간 질문을 넣고 어떤 필드가 화면으로 넘어가는지 읽어 보세요.

In [1]:
documents = [{'id': 'RULE-A',
  'equipment': '장비 A',
  'text': '점검 전 전원 표시등을 확인합니다.',
  'source': '가상 장비실 규정 A',
  'page': 1,
  'section': '2.1',
  'revised_on': '2026-11-01',
  'version': 'rules-v1',
  'groups': ['public', 'admin']},
 {'id': 'RULE-B',
  'equipment': '장비 B',
  'text': '점검 전 고정 장치를 확인합니다.',
  'source': '가상 장비실 규정 B',
  'page': 2,
  'section': '2.2',
  'revised_on': '2026-11-01',
  'version': 'rules-v1',
  'groups': ['public', 'admin']},
 {'id': 'RULE-ADMIN',
  'equipment': None,
  'text': '관리자 전용 절차는 선임 검토 기록을 확인하는 것입니다.',
  'source': '가상 장비실 내부 규정',
  'page': 3,
  'section': '3.1',
  'revised_on': '2026-11-01',
  'version': 'rules-v1',
  'groups': ['admin']}]

def thin_lookup(question):
    if not isinstance(question, str) or not question.strip():
        return {"status": "invalid_input", "answer": "질문을 입력하세요.", "sources": []}
    matches = [d for d in documents if d["equipment"] and
               d["equipment"] in question and "public" in d["groups"]]
    if len(matches) != 1:
        return {"status": "need_clarification", "answer": "장비 A 또는 장비 B 중 하나를 지정하세요.", "sources": []}
    doc = matches[0]
    return {"status": "answered", "answer": doc["text"], "sources": [{"id": doc["id"]}]}

reply = thin_lookup("장비 A의 점검 전 확인 사항은?")
print(reply["answer"], reply["sources"])

점검 전 전원 표시등을 확인합니다. [{'id': 'RULE-A'}]


전원 표시등과 RULE-A가 찍힙니다. 아직 화면에서 2.1절이나 개정일을 찾을 수 없습니다. 88장에서 합의한 출처 기준을 충족하려면 응답에 그 값도 담아야 합니다.

## 같은 장비로 일정 질문 보내기

장비명은 맞지만 질문의 목적이 다른 경우입니다. 규정 A의 문장이 사실이어도 교체 일정의 근거로는 쓸 수 없습니다.



**볼 곳**　‘점검 규정’를 기준으로 양쪽의 ‘일정 질문’와 ‘어긋난 답’를 대조하세요.

**애니메이션의 의미**　같은 장비라도 일정 질문에는 점검 규정이 답이 되지 않습니다.


### 셀 02 · Q03의 응답 읽기

함수는 그대로 두고 질문만 바꿉니다.

인계 기록을 둘로 나눠 적으세요. Q01은 답 문장은 맞지만 출처의 절이 빠졌습니다. Q03은 교체 일정을 물었는데도 표시등 안내가 나왔습니다. 다음 팀원이 고칠 곳은 각각 출처 위치와 질문 의도입니다.

In [2]:
question = '장비 A를 다음 달에 교체하기로 확정했나요?'
print(thin_lookup(question)["answer"])

점검 전 전원 표시등을 확인합니다.


점검 문장이 또 나옵니다. 함수는 장비 이름을 발견한 뒤 ‘점검’과 ‘교체’ 중 무엇을 묻는지 보지 않았습니다. 이 응답은 다음 장의 실패 사례입니다.

### 과제 03 · 선택 연습 · 장비명 빠진 질문

어느 장비인지 묻는 경로로 들어가려면 어떤 입력을 주면 될까요? 상태값까지 확인하세요.

In [3]:
# thin_lookup에 장비명 없는 점검 질문을 넣어 보세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(thin_lookup(&quot;점검 전 확인 사항은?&quot;)[&quot;status&quot;])</code></pre>
<p>실행 결과</p><pre><code>need_clarification</code></pre>
<p>장비가 하나로 정해지지 않아 need_clarification이 나옵니다. 이때 답변을 추측하지 않고 장비 이름을 요청할 수 있습니다.</p>
</details>

## 돌아보기

Q03의 잘못된 답과 Q01의 빠진 출처 위치를 그대로 저장하세요. 92장에서 동일한 입력으로 어디까지 동작했는지 세어 봅니다.